# Mining Fleet Maintenance-Slot Optimiser: Experimental Benchmark Notebook
## Review 1 Milestone: 35% Project Completion

**Objective:**
Demonstrate that maintenance scheduling that accounts for **operational service demand** and **replacement-vehicle capacity** dramatically cuts service cancellations compared to traditional predictive/urgency-only heuristics under harsh mining conditions (extreme heat, dust, and intermittent GPS/sensor loss).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import pandas as pd
import numpy as np
from data_generator import generate_mining_fleet
from optimizer_core import TelemetryResilientPreprocessor, BaselineUrgencyScheduler, MultiObjectiveSlotOptimizer
from audit_logger import DispatcherAuditLogger

print("All modules imported successfully.")

### 1. Ingest Fleet & Harsh Operating Conditions
Simulate 24 Ultra-Class haul trucks (CAT 797F, Komatsu 930E) and excavators in open-pit environments with high dust (PM10 > 600 ug/m3), ambient temperatures up to 49C, and steep haul ramps.

In [ ]:
raw_data = generate_mining_fleet(num_vehicles=24, num_spares=4)
fleet_df = pd.DataFrame(raw_data["fleet"])
print(f"Fleet size: {len(fleet_df)} | Spares: {len(raw_data['spares'])} | Workshop Bays: {raw_data['workshop']['num_bays']}")
fleet_df[['vehicle_id', 'model', 'urgency_score', 'telemetry_status', 'shift_demand_tons', 'cancellation_cost_per_shift']].head(8)

### 2. Telemetry Degraded-Mode Analysis (GPS Loss & Sensor Dropouts)
Inspect the fleet for communication dropouts and verify resilient preprocessor safeguards.

In [ ]:
preprocessor = TelemetryResilientPreprocessor()
processed_fleet = [preprocessor.process_vehicle_telemetry(v) for v in raw_data["fleet"]]
proc_df = pd.DataFrame(processed_fleet)
proc_df[['vehicle_id', 'telemetry_status', 'location_confidence', 'transit_time_buffer_hours', 'operational_risk_multiplier']].drop_duplicates(subset=['telemetry_status'])

### 3. Baseline vs. Multi-Objective Optimization Experiment
Compare traditional FIFO Urgency-only scheduling vs. Proposed Multi-Objective Slot Optimizer with Standby Replacement Vehicle Pairing.

In [ ]:
baseline = BaselineUrgencyScheduler()
optimizer = MultiObjectiveSlotOptimizer(w_cancellation=0.5, w_breakdown_risk=0.5)

base_res = baseline.schedule(raw_data["fleet"], workshop_bays=3)
opt_res = optimizer.optimize(raw_data["fleet"], raw_data["spares"], raw_data["workshop"])

comparison = {
    "Metric": ["Maintenance Scheduled", "Actual Service Cancellations", "Cancellation Cost ($)", "Direct Revenue Saved ($)"],
    "Baseline (FIFO)": [base_res["scheduled_maintenance_count"], base_res["service_cancellations_count"], f"${base_res['total_cancellation_cost_usd']:,.2f}", "$0.00"],
    "Multi-Objective Optimizer": [opt_res["scheduled_maintenance_count"], opt_res["actual_service_cancellations"], f"${opt_res['total_cancellation_cost_usd']:,.2f}", f"${base_res['total_cancellation_cost_usd'] - opt_res['total_cancellation_cost_usd']:,.2f}"]
}
pd.DataFrame(comparison)

### 4. Edge Cases and Stress Testing
- **Edge Case 1:** Severe Dust Storm (35% Telemetry Loss)
- **Edge Case 2:** Zero Spares Available (Spare Capacity Starvation)
- **Edge Case 3:** 15-Truck Critical Urgency Surge (Workshop Saturation)

In [ ]:
# Edge Case 2: Zero Spares
opt_zero_spares = optimizer.optimize(raw_data["fleet"], [], raw_data["workshop"])
print("Zero Spares Handled:", opt_zero_spares['actual_service_cancellations'], "cancellations (expected gracefully).")

# Edge Case 3: Urgency Surge
surge_fleet = [dict(v) for v in raw_data["fleet"]]
for v in surge_fleet[:15]:
    v["urgency_score"] = 95.0
opt_surge = optimizer.optimize(surge_fleet, raw_data["spares"], raw_data["workshop"])
print("Bay Limit Enforced:", opt_surge['scheduled_maintenance_count'], "units slotted into 3 bays (Hard constraint intact).")

### 5. Authorized Dispatcher Override & Immutable Audit Hash Chain
Simulate authorized operational override and cryptographically verify audit log.

In [ ]:
audit_logger = DispatcherAuditLogger(log_filepath="../dispatcher_override_audit.jsonl")
override_entry = audit_logger.record_dispatcher_override(
    plan_id="NOTEBOOK-PLAN-01",
    vehicle_id="VEH-008",
    original_slot="BAY-01_SHIFT-01",
    new_slot="DEFERRED_BENCH_BLAST",
    dispatcher_id="DISP-SUPERINTENDENT",
    dispatcher_role="Chief Mining Operations",
    override_reason="Blasting evacuation clearance required on Bench 9",
    service_impact_acknowledged=True
)
print("Override logged successfully:", override_entry["current_hash"])
print("Audit Chain Integrity Check:", audit_logger.verify_audit_integrity())